# LeetCode #201 — Bitwise AND of Numbers Range: tiền tố nhị phân chung

**Mức độ:** Medium · **Chủ đề:** Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def rangeBitwiseAnd(self, left: int, right: int) -> int:
```

Mục tiêu của notebook:

1. Vì sao không thể AND lần lượt mọi số (khoảng có thể dài `2^31`).
2. Quan sát: kết quả là **tiền tố nhị phân chung** của `left` và `right`, phần sau toàn 0.
3. Hai cách `O(log)`: **dịch phải** cả hai tới khi bằng nhau; hoặc **xóa bit 1 thấp nhất** của `right` (`right & (right − 1)`) tới khi `right <= left`.
4. Mở rộng: OR / XOR của một khoảng.

# 1. Bài toán là gì?

Cho hai số `left` và `right` biểu diễn khoảng `[left, right]`, trả về phép **AND bit** của **mọi** số trong khoảng (tính cả hai đầu).

**Constraints (theo đề):**

- `0 <= left <= right <= 2^31 - 1`

---

## Ví dụ

```text
left = 5, right = 7               ->  4      (101 & 110 & 111 = 100)
left = 0, right = 0               ->  0
left = 1, right = 2147483647      ->  0
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Biểu diễn nhị phân và các phép toán bit

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [1]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [2]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [3]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [4]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [5]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.2 Áp dụng vào bài này

AND của mọi số trong `[left, right]` giữ lại đúng phần **tiền tố nhị phân chung** của `left` và `right`: từ bit khác nhau đầu tiên trở xuống, trong khoảng chắc chắn có một số có bit đó bằng 0. Tìm tiền tố chung bằng **dịch phải** cả hai đến khi bằng nhau rồi dịch trái lại, hoặc xóa bit thấp nhất của `right` (`right & (right − 1)`) đến khi `right <= left`.

# 3. Nền tảng

In [6]:
import random, time
from functools import reduce

for x in [5, 6, 7]:
    print(f"{x} = {x:03b}")
print("5 & 6 & 7 =", 5 & 6 & 7, "=", format(5 & 6 & 7, "03b"))

5 = 101
6 = 110
7 = 111
5 & 6 & 7 = 4 = 100


Nhắc lại: AND của nhiều số có bit `k` bằng 1 **khi và chỉ khi** **mọi** số đều có bit `k` bằng 1.

# 4. Baseline: AND lần lượt

In [7]:
def range_and_brute(left: int, right: int) -> int:
    res = left
    for x in range(left + 1, right + 1):
        res &= x
        if res == 0:
            break                          # đã 0 thì AND tiếp vẫn là 0
    return res

print(range_and_brute(5, 7), range_and_brute(0, 0), range_and_brute(1, 2**31 - 1))

4 0 0


Dừng sớm khi kết quả bằng 0 giúp nhiều trường hợp, nhưng khoảng như `[2^30, 2^31 − 1]` có kết quả khác 0 (`2^30`) và dài `2^30` số → vẫn quá chậm.

# 5. Ý tưởng chính: tiền tố chung

Viết `left` và `right` dạng nhị phân **cùng độ dài**. Tìm vị trí bit **cao nhất** mà chúng **khác nhau** (tại đó `left` có 0, `right` có 1).

```text
left  = 26 = 1 1 0 1 0
right = 30 = 1 1 1 1 0
             └─┘ ↑
         tiền tố chung, bit khác nhau đầu tiên
```

- Các bit **trước** vị trí đó giống nhau ở `left` và `right` — và cũng giống ở **mọi** số giữa chúng (mọi số trong khoảng có chung tiền tố này) → giữ nguyên.
- Từ vị trí đó trở xuống: khi đếm từ `left` tới `right`, có lúc đi qua số dạng `tiền_tố 1 000…0` và số ngay trước nó `tiền_tố 0 111…1` → mọi bit từ vị trí đó trở xuống đều có ít nhất một số bằng 0 → kết quả là 0.

Vậy **đáp án = tiền tố chung, phần còn lại điền 0**.

In [8]:
for l, r in [(26, 30), (5, 7), (12, 15), (8, 8)]:
    w = max(r.bit_length(), 1)
    print(f"[{l:>2}, {r:>2}]: left = {l:0{w}b}, right = {r:0{w}b} → AND = {range_and_brute(l, r):0{w}b}")

[26, 30]: left = 11010, right = 11110 → AND = 11000
[ 5,  7]: left = 101, right = 111 → AND = 100
[12, 15]: left = 1100, right = 1111 → AND = 1100
[ 8,  8]: left = 1000, right = 1000 → AND = 1000


## Cách 1: dịch phải tới khi bằng nhau

Dịch phải cả hai, đếm số lần dịch `shift`, cho tới khi `left == right` (lúc đó chỉ còn tiền tố chung). Dịch trái trở lại `shift` lần.

In [9]:
def range_and_shift(left: int, right: int) -> int:
    shift = 0
    while left < right:
        left >>= 1
        right >>= 1
        shift += 1
    return left << shift

print(range_and_shift(5, 7), range_and_shift(26, 30), range_and_shift(1, 2**31 - 1))

4 24 0


## Cách 2: xóa dần bit 1 thấp nhất của `right` (mẹo Brian Kernighan)

`x & (x − 1)` xóa bit 1 **thấp nhất** của `x`. Lặp `right &= right − 1` chừng nào `right > left`: mỗi lần bỏ một bit 1 ở phần "đuôi" của `right`; khi `right <= left`, những gì còn lại là tiền tố chung (điền 0 phía sau).

In [10]:
def range_and_kernighan(left: int, right: int) -> int:
    while right > left:
        right &= right - 1
    return right

print(range_and_kernighan(5, 7), range_and_kernighan(26, 30), range_and_kernighan(1, 2**31 - 1))

4 24 0


# 6. Mô phỏng

In [11]:
def trace_shift(left, right):
    w = max(right.bit_length(), 1)
    shift = 0
    print(f"bắt đầu: left = {left:0{w}b}, right = {right:0{w}b}")
    while left < right:
        left >>= 1; right >>= 1; shift += 1
        print(f"dịch {shift}: left = {left:0{w - shift}b}, right = {right:0{w - shift}b}")
    print(f"bằng nhau → tiền tố {left:b}, dịch trái {shift} → {left << shift:0{w}b} = {left << shift}")

def trace_kernighan(left, right):
    w = max(right.bit_length(), 1)
    print(f"left = {left:0{w}b}")
    while right > left:
        nr = right & (right - 1)
        print(f"   right = {right:0{w}b} → right & (right-1) = {nr:0{w}b}")
        right = nr
    print(f"→ {right:0{w}b} = {right}")

trace_shift(26, 30)
trace_kernighan(26, 30)

bắt đầu: left = 11010, right = 11110
dịch 1: left = 1101, right = 1111
dịch 2: left = 110, right = 111
dịch 3: left = 11, right = 11
bằng nhau → tiền tố 11, dịch trái 3 → 11000 = 24
left = 11010
   right = 11110 → right & (right-1) = 11100
   right = 11100 → right & (right-1) = 11000
→ 11000 = 24


# 7. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def rangeBitwiseAnd(self, left: int, right: int) -> int:
        shift = 0
        while left < right:
            left >>= 1
            right >>= 1
            shift += 1
        return left << shift

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.rangeBitwiseAnd(5, 7), 4)
check("Ví dụ 2", sol.rangeBitwiseAnd(0, 0), 0)
check("Ví dụ 3", sol.rangeBitwiseAnd(1, 2**31 - 1), 0)
check("Bằng nhau", sol.rangeBitwiseAnd(13, 13), 13)
check("Nửa trên", sol.rangeBitwiseAnd(2**30, 2**31 - 1), 2**30)

random.seed(201)
for _ in range(5000):
    l = random.randint(0, 2000)
    r = l + random.randint(0, random.choice([3, 50, 1000]))
    exp = reduce(lambda a, b: a & b, range(l, r + 1))
    assert sol.rangeBitwiseAnd(l, r) == exp == range_and_kernighan(l, r) == range_and_brute(l, r), (l, r)
print("5000 khoảng ngẫu nhiên: khớp AND trực tiếp")

t0 = time.perf_counter()
for _ in range(10**5):
    sol.rangeBitwiseAnd(random.randint(0, 2**30), 2**31 - 1)
t1 = time.perf_counter()
print(f"10^5 truy vấn lớn: {t1 - t0:.3f}s (mỗi truy vấn ≤ 31 vòng)")

Ví dụ 1          PASS  4
Ví dụ 2          PASS  0
Ví dụ 3          PASS  0
Bằng nhau        PASS  13
Nửa trên         PASS  1073741824


5000 khoảng ngẫu nhiên: khớp AND trực tiếp

10^5 truy vấn lớn: 0.598s (mỗi truy vấn ≤ 31 vòng)


# 8. Vì sao đúng? Độ phức tạp?

Gọi `p` là vị trí bit cao nhất mà `left` và `right` khác nhau (`left` có 0, `right` có 1 vì `left < right`). Số `m = (tiền tố) 1 000…0` (bit `p` bằng 1, các bit thấp hơn bằng 0) thỏa `left < m <= right`, và `m − 1 = (tiền tố) 0 111…1` cũng nằm trong khoảng. Hai số này có các bit `0..p` **bù nhau** → AND tại các bit đó bằng 0. Các bit cao hơn `p` thì mọi số trong khoảng đều bằng tiền tố chung (vì `left` và `right` có chung tiền tố, và mọi số kẹp giữa cũng vậy). Kết luận: AND = tiền tố chung, các bit `0..p` bằng 0.

- **Dịch phải** dừng đúng khi chỉ còn tiền tố chung.
- **Kernighan:** chừng nào bit `p` của `right` còn bằng 1 thì `right > left` (cùng tiền tố, nhưng ở bit `p` thì `right` có 1 còn `left` có 0) → vòng lặp tiếp tục, xóa dần các bit 1 thấp nhất (đều nằm ở vị trí `<= p`). Bit `p` bị xóa ở lượt cuối cùng; khi đó `right` = tiền tố chung + toàn 0, và `right <= left` → dừng đúng đáp án.

```text
Time:  O(log right) ≤ 31 vòng
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: AND lần lượt mọi số (mục 4)

In [14]:
t0 = time.perf_counter(); range_and_brute(2**30, 2**30 + 2 * 10**6); t1 = time.perf_counter()
print(f"chỉ 2·10^6 số đã mất {t1 - t0:.2f}s; khoảng [2^30, 2^31 − 1] dài ~10^9 số")

chỉ 2·10^6 số đã mất 0.21s; khoảng [2^30, 2^31 − 1] dài ~10^9 số


## Lỗi 2: Chỉ AND hai đầu `left & right`

In [15]:
for l, r in [(5, 7), (8, 15), (4, 7)]:
    print(f"[{l}, {r}]: left & right = {l & r} | đúng = {sol.rangeBitwiseAnd(l, r)}")

[5, 7]: left & right = 5 | đúng = 4
[8, 15]: left & right = 8 | đúng = 8
[4, 7]: left & right = 4 | đúng = 4


`5 & 7 = 5` nhưng `6` có bit 0 bằng 0 → đáp án `4`. Các số ở giữa có thể xóa thêm bit.

## Lỗi 3: Kernighan với điều kiện `right >= left`

In [16]:
def wrong_kernighan(left, right, limit=64):
    steps = 0
    while right >= left and steps < limit:        # SAI: khi right == left vẫn tiếp tục xóa bit
        right &= right - 1
        steps += 1
    return right

for l, r in [(5, 7), (13, 13)]:
    print(f"[{l}, {r}]: sai = {wrong_kernighan(l, r)} | đúng = {sol.rangeBitwiseAnd(l, r)}")

[5, 7]: sai = 4 | đúng = 4
[13, 13]: sai = 12 | đúng = 13


Khi `right == left` đã là tiền tố chung, xóa thêm bit sẽ làm mất bit đúng (`[13, 13]` phải là 13). Với `left = 0` và `right = 0`, `right >= left` luôn đúng → không bao giờ dừng (ở đây chặn bằng `limit`).

## Lỗi 4: Quên dịch trái trở lại

Bản dịch phải trả `left` (chỉ là tiền tố đã bị dịch) thay vì `left << shift` → `[5, 7]` cho `1` thay vì `4`.

# 10. Mẹo nhận diện pattern

- **Phép bit trên cả một khoảng số** → nghĩ theo **từng bit**: bit `k` thay đổi theo chu kỳ `2^(k+1)` khi đếm lên.
- **AND làm mất bit dễ, OR làm bật bit dễ** → AND khoảng = tiền tố chung + 0; OR khoảng = tiền tố chung + toàn 1 (bài tập 4).
- **`x & (x − 1)`** xóa bit 1 thấp nhất — dùng đếm bit, kiểm tra lũy thừa 2, v.v.

Bài liên quan: **#191 Number of 1 Bits** (Easy), **#231 Power of Two** (Easy), **#338 Counting Bits** (Easy), **#1486 XOR Operation in an Array** (Easy), **#2275**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính AND của `[12, 15]` và `[17, 23]` bằng cách tìm tiền tố chung.

## Bài 2 — Điền chỗ trống (dịch phải)

In [17]:
def exercise_2_range_and(left: int, right: int) -> int:
    shift = 0
    # TODO
    pass


# assert exercise_2_range_and(5, 7) == 4
# assert exercise_2_range_and(0, 0) == 0
# assert exercise_2_range_and(1, 2**31 - 1) == 0
# assert exercise_2_range_and(13, 13) == 13

## Bài 3 — Viết bản Kernighan

In [18]:
def exercise_3_range_and_kernighan(left: int, right: int) -> int:
    # TODO
    pass


# assert exercise_3_range_and_kernighan(5, 7) == 4
# assert exercise_3_range_and_kernighan(26, 30) == 24
# assert exercise_3_range_and_kernighan(0, 0) == 0

## Bài 4 — Biến thể: OR của mọi số trong `[left, right]`

Gợi ý: tiền tố chung giữ nguyên, các bit từ vị trí khác nhau đầu tiên trở xuống đều thành **1**.

In [19]:
def exercise_4_range_or(left: int, right: int) -> int:
    # TODO
    pass


# assert exercise_4_range_or(5, 7) == 7
# assert exercise_4_range_or(8, 9) == 9
# assert exercise_4_range_or(4, 8) == 15
# assert exercise_4_range_or(3, 3) == 3

## Bài 5 — Mini challenge: XOR của mọi số trong `[left, right]` trong `O(1)`

Gợi ý: `f(n) = 0 ^ 1 ^ … ^ n` có chu kỳ 4 theo `n % 4`: `n, 1, n + 1, 0`. Khi đó XOR của khoảng là `f(right) ^ f(left − 1)` (vì `x ^ x = 0`).

In [20]:
def exercise_5_range_xor(left: int, right: int) -> int:
    # TODO
    pass


# assert exercise_5_range_xor(3, 9) == 3 ^ 4 ^ 5 ^ 6 ^ 7 ^ 8 ^ 9
# assert exercise_5_range_xor(0, 0) == 0
# assert exercise_5_range_xor(10**9, 10**9 + 3) == (10**9) ^ (10**9 + 1) ^ (10**9 + 2) ^ (10**9 + 3)

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_shift` sau khi tự làm.

**Bài 4**
- Mức 1: dịch phải đếm `shift` như bài gốc; kết quả `(left << shift) | ((1 << shift) − 1)`.

**Bài 5**
- Mức 1: `def f(n): return [n, 1, n + 1, 0][n % 4]` (với `n >= 0`); `f(−1) = 0`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
trace_shift(12, 15)
trace_shift(17, 23)

bắt đầu: left = 1100, right = 1111
dịch 1: left = 110, right = 111
dịch 2: left = 11, right = 11
bằng nhau → tiền tố 11, dịch trái 2 → 1100 = 12
bắt đầu: left = 10001, right = 10111
dịch 1: left = 1000, right = 1011
dịch 2: left = 100, right = 101
dịch 3: left = 10, right = 10
bằng nhau → tiền tố 10, dịch trái 3 → 10000 = 16


## Đáp án Bài 2

In [22]:
def answer_2_range_and(left: int, right: int) -> int:
    return range_and_shift(left, right)

assert answer_2_range_and(5, 7) == 4
assert answer_2_range_and(0, 0) == 0
assert answer_2_range_and(1, 2**31 - 1) == 0
assert answer_2_range_and(13, 13) == 13
print("Bài 2 PASS (giống range_and_shift ở mục 5)")

Bài 2 PASS (giống range_and_shift ở mục 5)


## Đáp án Bài 3

In [23]:
def answer_3_range_and_kernighan(left: int, right: int) -> int:
    return range_and_kernighan(left, right)

assert answer_3_range_and_kernighan(5, 7) == 4
assert answer_3_range_and_kernighan(26, 30) == 24
assert answer_3_range_and_kernighan(0, 0) == 0
print("Bài 3 PASS (giống range_and_kernighan ở mục 5)")

Bài 3 PASS (giống range_and_kernighan ở mục 5)


## Đáp án Bài 4

In [24]:
def answer_4_range_or(left: int, right: int) -> int:
    shift = 0
    while left < right:
        left >>= 1; right >>= 1; shift += 1
    return (left << shift) | ((1 << shift) - 1)

assert answer_4_range_or(5, 7) == 7
assert answer_4_range_or(8, 9) == 9
assert answer_4_range_or(4, 8) == 15
assert answer_4_range_or(3, 3) == 3
for _ in range(3000):
    l = random.randint(0, 500); r = l + random.randint(0, 300)
    assert answer_4_range_or(l, r) == reduce(lambda a, b: a | b, range(l, r + 1))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [25]:
def xor_upto(n: int) -> int:
    if n < 0:
        return 0
    return [n, 1, n + 1, 0][n % 4]

def answer_5_range_xor(left: int, right: int) -> int:
    return xor_upto(right) ^ xor_upto(left - 1)

assert answer_5_range_xor(3, 9) == 3 ^ 4 ^ 5 ^ 6 ^ 7 ^ 8 ^ 9
assert answer_5_range_xor(0, 0) == 0
assert answer_5_range_xor(10**9, 10**9 + 3) == (10**9) ^ (10**9 + 1) ^ (10**9 + 2) ^ (10**9 + 3)
for _ in range(3000):
    l = random.randint(0, 500); r = l + random.randint(0, 300)
    assert answer_5_range_xor(l, r) == reduce(lambda a, b: a ^ b, range(l, r + 1))
print("Bài 5 PASS")
print("chu kỳ f(n) với n = 0..11:", [xor_upto(n) for n in range(12)])

Bài 5 PASS
chu kỳ f(n) với n = 0..11: [0, 1, 3, 0, 4, 1, 7, 0, 8, 1, 11, 0]


# 14. Tổng kết

> AND của `[left, right]` = **tiền tố nhị phân chung** của `left` và `right`, các bit sau điền 0. Dịch phải cả hai tới khi bằng nhau rồi dịch trái lại; hoặc `while right > left: right &= right − 1`. `O(log right)`.

---

## Checklist trước khi nộp

- [ ] Không lặp qua mọi số trong khoảng?
- [ ] Dịch trái trở lại `shift` lần?
- [ ] Kernighan dùng `right > left` (không `>=`)?
- [ ] `left == right` → trả chính nó?